# 02. Startup Scenario and Vacuum Fields

A tokamak discharge does not simply happen. Coils are charged, gas is admitted, a loop
voltage is induced, and if the field at the moment of breakdown has the right shape, a
plasma forms and carries current. This session reads one VEST discharge backwards: from the
signals that show what the plasma did, to the fields that let it start at all — and then
repeats the same reading on three discharges side by side.

## Session Overview

By the end of this session you will be able to:

- read the plasma-response signals of a discharge — current, light, diamagnetic flux, loop
  voltage, local field — and see the discharge on the fast camera;
- find the breakdown time **once**, from the data, and reuse that one number everywhere;
- name what each actuator did: the TF field, the PF coils, the gas fill and the 2.45 GHz EC
  power;
- explain how an axisymmetric Green function becomes a mutual-inductance matrix, a vessel
  eddy-current solve and a vacuum-field map;
- check that the vacuum-field model reproduces the measured magnetics before trusting
  anything derived from it;
- read the reduced startup proxies — $V_{\mathrm{loop}}$, $B_z$, the decay index, the Lloyd
  margin, the connection length — as the proxies they are;
- run the same analysis on several shots and compare them in one table.

**Part I** (Guided Analysis) works through one discharge; **Part II** (Integrated Analysis)
repeats it for three. Everything runs offline on packaged VEST discharges; every cell is
written so that the same code runs on any shot loaded from the database.

## Physical Context

### Getting a plasma started

Startup has three phases, and each leaves a different fingerprint in the diagnostics.

**Breakdown.** The central solenoid swings, inducing a toroidal electric field. Free
electrons accelerate, ionise the fill gas, and an avalanche runs away. Whether it does turns
on three numbers together — the field, the fill pressure, and how far a line runs before it
meets a wall — and the last of them is why the field null matters. The session works all
three out.

**Burn-through.** The young plasma is cold and full of neutrals and impurities, which radiate
away much of the ohmic input. It either heats through this barrier or it dies. Line
radiation, which the filterscopes see, is the direct evidence.

**Current ramp-up.** Past burn-through the plasma is conducting well, and the transformer
drives its current up.

### The vacuum field, and why it is a *precondition*

Before any plasma exists, the coils and the vessel already make a field. Three properties of
it decide whether startup is possible: the **loop voltage**, which supplies the drive; the
**vertical field** $B_z$, which balances the plasma's outward hoop force once current flows;
and the **decay index**, which decides whether that balance is stable rather than merely
present. Each is written down where the session computes it, beside the figure it explains.

### The vessel is part of the circuit

VEST's vacuum vessel is a conductor. A changing coil current induces eddy currents in it,
and those currents make a field of their own. No honest vacuum-field calculation can leave
them out — and the packaged discharge does not record them, so the session solves for them
before it reads the vacuum field at all.

## Load / Prepare Data

### The discharge

`SHOT` is the only place a shot number is written in Part I. The packaged sample runs
offline; the commented line below it is the lab-mode path, which loads any VEST discharge
from the database (it needs HSDS credentials). Nothing after this cell assumes 39915: times
come from the data, and a cell whose input a discharge does not carry says so and moves on.

In [ ]:
import copy

import numpy as np
import matplotlib.pyplot as plt
import vaft
from vaft.machine_mapping.wall import wall as vest_wall

In [ ]:
SHOT = 39915
ods = vaft.omas.sample_ods(SHOT)
# ods = vaft.database.load(SHOT)  # lab mode: any shot, needs HSDS credentials

# Every vacuum-field cell reads the limiter outline. A database product may carry none, or a
# partial one from an older processing era, so fill it from VAFT's packaged VEST geometry
# (for the packaged sample this rewrites the outline it already has).
vest_wall(ods)

# The startup reference point: VEST's on-axis reference radius, stored with the TF product.
r_ref = float(ods["tf.r0"])
print(f"shot {SHOT}: {', '.join(sorted(ods.keys()))}")
print(f"startup reference point: R = {r_ref:.2f} m, Z = 0")

## Guided Analysis

### Part I, step 1 — what the plasma did

Start with the response signals, one figure each. The plasma current says whether there was a
plasma; the line emission says what state it was in; the diamagnetic flux says whether it
held pressure; the loop voltage and the local field say what the machine was doing to it.

In [ ]:
vaft.omas.plot_plasma_current_time(ods)
plt.show()

In [ ]:
vaft.omas.plot_spectrometer_uv_time_intensity(ods, emission="H_alpha")
plt.show()

H-alpha is neutral hydrogen at the edge — recycling and fuelling. Impurity lines tell the
burn-through story instead: carbon and oxygen come off the wall, radiate, and have to be
overcome. Do not pick one line (`emission="CIII"` draws carbon alone); `emission="all"` draws
every processed line of every filterscope channel, hydrogen Balmer lines, C II/C III and
O I/O II/O V together, so the ionisation stages can be compared on one axis.

In [ ]:
vaft.omas.plot_spectrometer_uv_time_intensity(ods, emission="all", legend=True)
plt.show()

In [ ]:
vaft.omas.plot_diamagnetic_flux_time(ods)
plt.show()

The loop voltage measured by a flux loop is the rate of change of the flux it links. The
inboard-midplane loop (`selection="inboard_mid"`, the inboard loop nearest $Z = 0$) sits closest
to where the plasma will form. Its sign and its factor of $2\pi$ depend on whether a flux is
stored per weber or per radian, and VAFT's kernels do not yet agree on either
([#354](https://github.com/VEST-Tokamak/vaft/issues/354)) — read the shape and the timing of this
trace before its sign.

In [ ]:
vaft.omas.plot_flux_loop_time_voltage(ods, selection="inboard_mid")
plt.show()

In [ ]:
vaft.omas.plot_b_field_probe_time_field(ods, selection="inboard_mid")
plt.show()

### Step 2 — see the discharge

Before any physics, look at it. The fast camera recorded the discharge directly, and 66 of its
frames from shot 39915 ship with the VAFT repository — not with the installed package, so this
part needs a clone. They are not in `sample_ods()` either; the cell below writes a handful of
them into the `camera_visible` IDS, which is the form every camera plot reads. On a shot without
packaged frames the camera cells say so and skip.

In [ ]:
import cv2
from vaft.machine_mapping.camera_visible import (
    vfit_camera_visible_dynamic,
    vfit_camera_visible_static,
)

camera_frames = []
if "camera_visible" in ods:
    print("camera_visible is already in the ODS")
else:
    try:
        camera_frames = vaft.data.sample_camera_visible_frame_paths(SHOT)
    except FileNotFoundError as error:
        print(f"no packaged camera frames for shot {SHOT}: {error}")

if camera_frames:
    # Every eleventh packaged frame: six across the discharge. OMAS stores a frame as int64,
    # about 10 MB each, so all 66 would be most of a gigabyte for a slider that needs a few.
    shown = camera_frames[::11]
    images = [cv2.imread(str(path), cv2.IMREAD_GRAYSCALE) for _, path in shown]
    vfit_camera_visible_static(
        ods, lines_n=images[0].shape[0], columns_n=images[0].shape[1], channel_name="Fast Camera",
    )
    vfit_camera_visible_dynamic(ods, images=images, times_s=[time_s for time_s, _ in shown])
    print(f"{len(images)} frames: {', '.join(f'{time_s * 1e3:.1f}' for time_s, _ in shown)} ms")
has_camera = "camera_visible" in ods

In [ ]:
# interaction_backend="auto" gives a live frame slider in Jupyter: drag it and the discharge
# plays back. This cell steps the same control from code, so it also runs headless, and
# leaves the last frame on screen.
if has_camera:
    result = vaft.omas.plot_camera_visible_image(ods, interactive=True, interaction_backend="none")
    slider = next(control for control in result.controls if control.name == "frame_index")
    print(f"{slider.label}: {slider.options[1] + 1} frames")
    for index in range(slider.options[1] + 1):
        result.state.set("frame_index", index)
        print(f"  {result.axes.get_title()}")
    plt.show()
else:
    print("no camera_visible frames for this shot: nothing to play back")

The six frames span about twenty milliseconds of the discharge, from just before the onset the
light will give below to the current decay; the first is washed out, and from the next one on the
vessel is lit from inside. The bright vertical band is
the centre stack, not plasma. Keep these pictures in mind: everything below explains *why* the
vessel lit up when it did.

### Step 3 — when did the plasma form?

Reading a breakdown time off an H-alpha trace by eye is a habit worth breaking. The onset can be
found from the data, and it is found **once**: every later cell of Part I uses this one
`t_breakdown`.

In [ ]:
t_breakdown = vaft.omas.find_breakdown_onset(ods)
print(f"breakdown onset: {t_breakdown * 1e3:.2f} ms")

That one number is not a threshold read off one trace. `find_breakdown_onset` is a thin wrapper
over `plasma_timing`, and the object it throws away is where the criterion lives:

- **H-alpha, chosen by label, is authoritative.** The detector asks for the line called
  `H-alpha_6563`, never for a channel number, and prefers the slowest digitizer that carries it.
  Light is optical, so the coil-firing pickup every magnetic diagnostic carries cannot trigger it.
- **The plasma current is the fallback, and always the cross-check.** It is computed whether or
  not the light answered, and the verdict records whether the two agree.
- **A raw magnetic first crossing is never used.** It fires on the coils, not on the plasma.

The test is not a derivative. A sample has to exceed
$\text{baseline} + \max(f\cdot\text{peak},\ \sigma\cdot\sigma_{\text{robust}})$, *stay* above it for
half a millisecond, and belong to a run wide, prominent and large enough to be a pulse rather
than a spike. The fraction-of-peak term is what makes two channels with a ten-fold difference in
noise agree on the onset.

In [ ]:
from vaft.omas.plasma_timing import plasma_timing
from vaft.omas.discharge_timing import discharge_timing

timing = plasma_timing(ods)
events = discharge_timing(ods)

# Any of these can be None on a shot that lacks a signal (no filterscope, no
# usable current, no OH coil trace), so each line says so instead of failing.
ms = lambda value: "   n/a " if value is None else f"{value * 1e3:7.2f}"
print(f"ohmic coil fired   : {ms(events.oh_onset)} ms  ({events.oh_coil})")
print(f"loop voltage zero  : {ms(events.vloop_time)} ms")
print(f"breakdown onset    : {ms(t_breakdown)} ms")
print(f"plasma window      : {ms(timing.onset)} - {ms(timing.offset)} ms")
print(f"decided by         : {timing.source}")
if timing.onset_delta_s is None:
    print(f"light and current  : {timing.agreement}; only one of them gave an onset")
else:
    print(f"light and current  : {timing.agreement}; the current started "
          f"{abs(timing.onset_delta_s) * 1e6:.0f} us "
          f"{'before' if timing.onset_delta_s < 0 else 'after'} the light")
if timing.duty_cycle is not None:
    print(f"above threshold    : {timing.duty_cycle:.0%} of the window")

Read it as a sequence. The ohmic coil fires; about ten milliseconds later the loop voltage
crosses zero; the gas breaks down about four milliseconds after that. The light and the current
agree to within a sample or two — `consistent` — and the window is above threshold for all of its length, so its duration
really is a duration and not an envelope drawn around gaps
([#752](https://github.com/VEST-Tokamak/vaft/issues/752)).

Now put the six response signals on one time axis with that onset drawn through all of them.

In [ ]:
figure, axes = plt.subplots(6, 1, sharex=True, figsize=(9, 15))
vaft.omas.plot_plasma_current_time(ods, ax=axes[0], show=False)
vaft.omas.plot_spectrometer_uv_time_intensity(ods, emission="H_alpha", ax=axes[1], show=False)
vaft.omas.plot_spectrometer_uv_time_intensity(ods, emission="all", legend=False, ax=axes[2], show=False)
vaft.omas.plot_diamagnetic_flux_time(ods, ax=axes[3], show=False)
vaft.omas.plot_flux_loop_time_voltage(ods, selection="inboard_mid", ax=axes[4], show=False)
vaft.omas.plot_b_field_probe_time_field(ods, selection="inboard_mid", ax=axes[5], show=False)
for axis in axes:
    axis.axvline(t_breakdown, color="k", ls="--", lw=0.8)
axes[0].set_title(f"#{SHOT}: response signals, breakdown onset at {t_breakdown * 1e3:.1f} ms")
figure.tight_layout()
plt.show()

Read down the dashed line. The loop voltage swung negative when the solenoid fired, crossed zero,
and is climbing through a couple of volts at the onset; the H-alpha light, the impurity lines and
the plasma current all leave their baselines at it, within a sample of one another; the diamagnetic
flux rises as soon as there is current to confine pressure; and the inboard-midplane $B_z$ probe,
which had been following the coils and dipped negative, crosses zero at the onset and climbs as the
plasma current adds its own field.

### Step 4 — what drove it

#### The toroidal field

The TF coil sets the field the plasma sits in. What VAFT stores is not $B_T$ but the product
$B_T R$ (`tf.b_field_tor_vacuum_r`): a vacuum toroidal field falls as $1/R$, so the product is a
constant of the coil current and $B_T(R) = (B_T R)/R$ anywhere in the vessel. A quoted "$B_T$"
therefore always needs a radius — on VEST it is the vessel's on-axis reference radius
$R_0$ = `tf.r0`, 0.4 m.

In [ ]:
vaft.omas.plot_tf_coil_time_current(ods)
plt.show()
# vaft.omas.plot_tf_coil_time_b_t(ods)  # the same waveform as B_T at R0 = tf.r0

tf_time = np.asarray(ods["tf.b_field_tor_vacuum_r.time"], dtype=float)
b_t_r = float(ods["tf.b_field_tor_vacuum_r.data"][int(np.argmin(np.abs(tf_time - t_breakdown)))])
print(f"at breakdown: B_T R = {b_t_r:.4f} T m, so B_T = {abs(b_t_r) / r_ref:.3f} T at R0 = {r_ref:.2f} m "
      f"and {abs(b_t_r) / (2 * r_ref):.3f} T at twice that radius")

#### The poloidal field coils

Ten PF circuits, each with a job in this configuration:

- **PF1** is the central solenoid — the Ohmic transformer's primary. Its swing induces the loop
  voltage that drives breakdown and the current ramp; it supplies *drive*, not "Ohmic power"
  directly.
- **PF5** shapes the field null the breakdown happens in.
- **PF6 and PF9/10** supply the vertical field for equilibrium — the radial force balance that
  holds a current ring at its major radius.

How the waveforms come about matters for reading them. VEST's PF currents are set by
**capacitor-bank discharge circuits**: the charging voltage, the power-supply switch timing, and
each coil circuit's inductance, resistance and capacitance decide the shape, and once a bank is
fired the waveform runs its course. Most tokamaks instead drive their coils from
**current-regulated supplies** — as VEST's own H-bridge controller does — where a feedforward
command programs the intended waveform (a flat-top, say) and feedback control corrects the
residual between it and the measured current. Feedforward alone does not make a flat-top; it is
the feedback loop that holds one.

In [ ]:
vaft.omas.plot_pf_coil_geometry_poloidal(ods)
plt.show()

In [ ]:
vaft.omas.plot_pf_coil_time_current(ods)
plt.show()
# vaft.omas.plot_pf_coil_time_current_turns(ods)  # ampere-turns = current x turns, what the field sees

#### The gas

The fill pressure is the third ingredient of breakdown. The gauge is displayed in Torr, the unit
the VEST log uses; pass `yunit="Pa"` for SI, and `vaft.formula.PA_PER_TORR` converts by hand.

In [ ]:
if "barometry" in ods:
    vaft.omas.plot_barometry_time_pressure(ods)
    # vaft.omas.plot_barometry_time_pressure(ods, yunit="Pa")
    plt.axvline(t_breakdown, color="k", ls="--", lw=0.8)
    plt.show()
    print(f"1 Torr = {vaft.formula.PA_PER_TORR:.2f} Pa")
else:
    print(f"shot {SHOT} carries no barometry: no fill pressure, and no Lloyd threshold below")

#### The 2.45 GHz EC power

VEST pre-ionises with a 6 kW, 2.45 GHz magnetron. Its forward and reflected power are recorded by
log detectors, and `ec_launchers` stores their difference — a **net launched-power estimate**, not
a measured absorbed power ([#165](https://github.com/VEST-Tokamak/vaft/issues/165)). A log detector
is spiky, so the trace is read through a rolling median (`smooth=`, in seconds); detector voltages
outside the calibrated range are masked rather than calibrated into nonsense. The legacy calibration
is not certified, and the estimate can read above the magnetron's rating: trust *when* the source
was on far more than *how many* kilowatts it delivered.

In [ ]:
if "ec_launchers" in ods:
    vaft.omas.plot_ec_launchers_time_power(ods, smooth=1e-3)
    plt.axvline(t_breakdown, color="k", ls="--", lw=0.8)
    plt.show()
    ec_power = np.asarray(ods["ec_launchers.beam.0.power_launched.data"], dtype=float)
    ec_time = np.asarray(ods["ec_launchers.beam.0.power_launched.time"], dtype=float)
    near = np.abs(ec_time - t_breakdown) <= 1e-3
    print(f"net EC power within 1 ms of breakdown (median): {np.nanmedian(ec_power[near]) / 1e3:.2f} kW")
else:
    print(f"shot {SHOT} carries no ec_launchers: EC pre-ionisation cannot be judged")
# vaft.machine_mapping.vest_ec_power(SHOT)  # lab mode: forward and reflected separately, from raw data

One actuator still belongs in this picture and is not here: the **gas valve command**. It is
recorded on VEST but not mapped into IMAS yet, so only its consequence — the gauge above — can be
plotted. Its absence is a mapping gap, not a quiet valve. The EC power, which used to be the second
such gap, is now mapped.

### Step 5 — solving the vessel currents

#### What the model is made of

At any moment the machine carries current in three places, and only one of them is the plasma.
The figure below is the model's inventory: the **PF coils**, which were driven; the **passive
structure** — the vessel and its supports cut into 950 axisymmetric loops, which respond; and
the **plasma**, drawn here as a few illustrative filaments around the startup reference point,
because a filament model is how a plasma current enters the same equations. The vacuum solve
below passes no plasma filaments at all.

In [ ]:
limiter_r = np.asarray(ods["wall.description_2d.0.limiter.unit.0.outline.r"], dtype=float)
limiter_z = np.asarray(ods["wall.description_2d.0.limiter.unit.0.outline.z"], dtype=float)
minor_radius = 0.5 * float(limiter_r.max() - limiter_r.min())

angle = np.linspace(0.0, 2 * np.pi, 12, endpoint=False)
filament_r = np.concatenate([[r_ref], r_ref + 0.5 * minor_radius * np.cos(angle)])
filament_z = np.concatenate([[0.0], 0.5 * minor_radius * np.sin(angle)])

figure, axes = plt.subplots(figsize=(6, 9))
vaft.omas.plot_machine_geometry_poloidal(ods, ax=axes, show=False)
axes.plot(filament_r, filament_z, "o", color="tab:purple", ms=4,
          label="plasma filaments (illustrative)")
axes.legend(loc="upper right", fontsize="small")
plt.show()
print(f"PF coils: {len(ods['pf_active.coil'])}, passive loops: {len(ods['pf_passive.loop'])}, "
      f"plasma filaments in the vacuum solve: 0")

#### From a circular filament to a mutual inductance

Everything in this model is a circular loop around the symmetry axis, so one function does all
the work: the **Green function** of a circular filament. A unit current in a loop at $(R', Z')$
produces, at $(R, Z)$, the poloidal flux

$$\psi(R, Z) = \mu_0\, I\, G(R, Z; R', Z'), \qquad
  G = \frac{\sqrt{R R'}}{k}\left[(2 - k^2)K(k) - 2E(k)\right], \qquad
  k^2 = \frac{4 R R'}{(R + R')^2 + (Z - Z')^2},$$

with $K$ and $E$ the complete elliptic integrals (`vaft.formula.greens_function_exact`; the flux
per ampere $\mu_0 G$ is `green_psi_exact`, in full weber, and the field per ampere
$(B_R, B_Z)$ is `green_br_bz_exact`). The flux one loop's current puts through another loop *is*
their **mutual inductance**, $M_{ij} = \psi_i / I_j = \mu_0 G(R_i, Z_i; R_j, Z_j)$ — pairwise, from
geometry alone (`vaft.formula.mutual_inductance`, which averages $G$ over a finite cross-section).
The diagonal is different: $G$ diverges when a filament is asked for its own flux, so a **self
inductance** needs the conductor's finite cross-section (`vaft.formula.self_inductance`).

The cell checks the first claim numerically on two real coils.

In [ ]:
coil_index = {str(ods[f"pf_active.coil.{i}.name"]): i for i in ods["pf_active.coil"]}
rect_1 = ods[f"pf_active.coil.{coil_index['PF1']}.element.0.geometry.rectangle"]
rect_5 = ods[f"pf_active.coil.{coil_index['PF5']}.element.0.geometry.rectangle"]

flux_per_ampere = vaft.formula.green_psi_exact(rect_5["r"], rect_5["z"], rect_1["r"], rect_1["z"])
m_filament = vaft.formula.mutual_inductance(rect_1["r"], rect_1["z"], 0.0, 0.0,
                                            rect_5["r"], rect_5["z"], 0.0, 0.0)
m_finite = vaft.formula.mutual_inductance(rect_1["r"], rect_1["z"], rect_1["width"], rect_1["height"],
                                          rect_5["r"], rect_5["z"], rect_5["width"], rect_5["height"])
l_self = vaft.formula.self_inductance(rect_5["r"], rect_5["width"], rect_5["height"])

print(f"mu0 G (PF1 turn -> PF5 turn)  : {flux_per_ampere:.4e} Wb/A")
print(f"mutual_inductance, filaments  : {m_filament:.4e} H   (ratio {m_filament / flux_per_ampere:.6f})")
print(f"mutual_inductance, rectangles : {m_finite:.4e} H")
print(f"self_inductance of one PF5 turn: {l_self:.4e} H, {l_self / m_finite:.0f}x the mutual")

The ratio is one: a mutual inductance between filaments is the Green function times $\mu_0$ and
nothing else, and the finite cross-section moves it by under a percent at this separation. The
self term is hundreds of times larger — a loop links far more of its own flux than of a distant
coil's.

#### The circuit equation

Stack those numbers into matrices — $\mathbf{L}_{vv}$ among the vessel loops (self terms on the
diagonal), $\mathbf{M}_{va}$ from the active coils to the vessel, $\mathbf{M}_{vp}$ from the plasma
filaments to the vessel — and Faraday's law for every vessel loop becomes

$$\mathbf{L}_{vv}\,\frac{\mathrm{d}\mathbf{I}_v}{\mathrm{d}t} + \mathbf{R}_v\,\mathbf{I}_v
  = -\,\mathbf{M}_{va}\,\frac{\mathrm{d}\mathbf{I}_a}{\mathrm{d}t}
    - \mathbf{M}_{vp}\,\frac{\mathrm{d}\mathbf{I}_p}{\mathrm{d}t},$$

with $\mathbf{R}_v$ the diagonal loop resistance. In the vacuum solve the plasma term is absent.
Two things follow from the shape of it. The drive is $\mathrm{d}\mathbf{I}_a/\mathrm{d}t$, not
$\mathbf{I}_a$, so a steady coil current induces nothing. And the homogeneous response decays on the
eigenvalues of $\mathbf{R}_v^{-1}\mathbf{L}_{vv}$ — the wall's own $L/R$ times — which is why the
vessel field lags the coils rather than following them.

The chain VAFT runs is **em_coupling** (the Green-function matrices, stored or computed)
$\rightarrow$ **impedance** ($\mathbf{L}_{vv}$, $\mathbf{R}_v$) $\rightarrow$ **eddy solve** (the
equation above, integrated) $\rightarrow$ **vacuum field** (coil + vessel currents, times $G$, on any
grid). Solving it is a modelling step rather than a measurement, so the packaged shot stores no
result; passing empty plasma filament lists selects the vacuum case.

In [ ]:
before = {row["name"] for row in vaft.omas.available_plots(ods)}
print("passive current available?", "passive_structure_time_current" in before)
print("pf_passive loops        :", len(ods["pf_passive.loop"]))

vaft.omas.compute_eddy_currents(ods, [], [])

unlocked = {row["name"] for row in vaft.omas.available_plots(ods)} - before
print("time points solved:", len(ods["pf_passive.time"]))
print("plots unlocked    :", sorted(unlocked))

Views appeared, and the reason is worth noticing: most of them are not about the vessel itself.
They are magnetics comparisons, and they became available because the vacuum-field model they rest
on now exists. (A database product that already carries solved passive currents answers `True` to
the first line; the solve then refreshes them in vacuum mode.)

In [ ]:
vaft.omas.plot_current_overview(ods)
plt.show()

The vessel current is large — comparable to the plasma current — and it is *induced*, so it
opposes whatever the coils just did. A magnetic measurement made outside the vessel sees the sum of
all three. That is the whole difficulty of magnetic reconstruction in one figure.

Vacuum fields superpose, which is the only reason that sum can be taken apart again:

$$\psi(R, Z) = \psi_{\text{coils}} + \psi_{\text{vessel}} + \psi_{\text{plasma}},
  \qquad \psi_i = \mu_0\,G(R, Z; R_i, Z_i)\,I_i .$$

Each $\mu_0 G$ is a *response per unit current* — geometry only. The first two terms are computable
once the currents are known; the third is what a reconstruction is after, and it is only ever
reached by subtracting the other two.

### Step 6 — where this machinery is used

The same Green-function matrices serve five different purposes, and it helps to keep them apart:

1. **Vessel eddy currents** — the solve just run.
2. **Coil and eddy contributions to every magnetic signal** — the synthetic probe and flux-loop
   signals the next step compares with the measured ones.
3. **Field maps from PF + vessel + plasma currents**, which MHD equilibrium reconstruction fits to
   the magnetics — session 03.
4. **Startup analysis before an equilibrium exists.** The vacuum field is then all there is, and it
   is read with Townsend avalanche theory and Lloyd's breakdown criterion, together with the
   empirical startup conditions later work established: a wide field null, a long connection
   length, and a figure of merit $E_\varphi B_\varphi / B_p$ above about **1000 V/m for Ohmic** and
   **100 V/m for ECH-assisted** startup (`vaft.formula.startup.LLOYD_FIGURE_OF_MERIT_OHMIC_V_PER_M`,
   `..._ECH_V_PER_M`).
5. **Reduced proxies even after a plasma forms**, read at the vessel's on-axis **startup reference
   point** $(R_0, 0)$ — a fixed point of the vessel, not the magnetic axis:
   - $B_z$ there is a **radial force-balance proxy**. A large-aspect-ratio circular ring of current
     $I_p$, radius $R_0$ and minor radius $a$ needs the Shafranov vertical field
     $$B_v \simeq -\frac{\mu_0 I_p}{4\pi R_0}\left[\ln\frac{8R_0}{a} + \frac{l_i}{2} + \beta_p
       - \frac{3}{2}\right]$$
     (`vaft.formula.startup.vertical_field_from_I_p_R0_a_beta_p_li`). It assumes a thin, circular,
     large-aspect-ratio ring — VEST is none of those — so it gives the size and sign to expect, not
     an equilibrium.
   - $V_{\mathrm{loop}}$ is a **proxy for the inductive drive**. $V_{\mathrm{loop}} I_p$ is *not*
     the Ohmic power dissipated in the plasma during ramp-up: part of it goes into the magnetic
     energy $\tfrac12 L_p I_p^2$ that the rising current is building.
   - The **decay index** $n = -(R/B_z)\,\partial B_z/\partial R$ is a **rigid-ring vertical-stability
     proxy**; a conducting wall close to the plasma modifies the real stability window.

### Step 7 — does the vacuum model match the machine?

Everything below this point is derived from the field model. Before trusting it, check it against
the magnetics that were actually measured. The next figure puts three curves on each channel:

- **measured** — what that probe or flux loop actually recorded;
- **coil-only** — the synthetic signal from `pf_active` alone;
- **coil + eddy** — the same with the vessel currents solved for above added in.

The gap between the second and the third *is* the vessel, drawn directly. It is also the error a
reconstruction inherits if the vessel is left out, and because it is induced it does not look like
noise: it looks like current, in the place a plasma would be. Where the third curve reaches the
first, the vacuum model is complete and what remains is plasma. Where it does not, the leftover is
either plasma or a defect in the model, and this figure alone cannot say which.

In [ ]:
vaft.omas.plot_magnetics_overview_vacuum(ods)
plt.show()

Before breakdown the coil + eddy curve should sit on the measurement; after it, the difference is
the plasma's own field. The residual figure draws that difference. It marks an "Ip onset" of its
own, and that is a **different detector** from `t_breakdown`: a five-sigma first crossing of each
signal's pre-plasma noise band, applied identically to every residual so that they can be compared
with one another. On another shot the two numbers need not agree.

In [ ]:
vaft.omas.plot_magnetics_overview_plasma_residual(ods)
plt.show()

#### From a Rogowski coil to $I_p$ and diamagnetic flux

Two of the response signals of step 1 are not measurements in the raw sense; each is the end of a
processing chain that starts at a physical Rogowski coil. VEST maps both sensors into
`magnetics.rogowski_coil`, at the sensor level:

- **Plasma-current Rogowski coil** (`rogowski_coil:plasma_current`) — the inner contour linking the
  plasma current *and* the currents induced in the tungsten limiter around the centre-stack wall.
  Chain: **raw DAQ voltage** $\rightarrow$ calibration $\rightarrow$ `rogowski_coil[0].current`
  $\rightarrow$ baseline removal, the shot-era **FL10 compensation** that subtracts a flux-loop proxy
  for the limiter current, and the shot-era sign $\rightarrow$ `magnetics.ip`.
- **Diamagnetic hi-sensitivity TF-current Rogowski coil** (`rogowski_coil:diamagnetic_tf_current`)
  — a sensitive coil on the TF current. Chain: **raw DAQ voltage** $\rightarrow$ integration and
  calibration $\rightarrow$ `rogowski_coil[1].current` $\rightarrow$ subtraction of a reference
  (no-plasma) waveform, so that only the plasma's diamagnetic change $\Delta I_{TF}$ remains
  $\rightarrow$ conversion to flux $\rightarrow$ `magnetics.diamagnetic_flux` [Wb].

The stored `rogowski_coil` currents are **calibrated sensor currents**, not raw DAQ voltages and
not the derived products; calling them "raw" would collapse two stages and hide exactly the
processing a reader needs to see.

In [ ]:
for index in ods["magnetics.rogowski_coil"]:
    sensor = ods[f"magnetics.rogowski_coil.{index}"]
    current = np.asarray(sensor["current.data"], dtype=float)
    print(f"rogowski_coil[{index}]: {sensor['name']} ({sensor['identifier']}): "
          f"{np.nanmin(current):.4g} to {np.nanmax(current):.4g} A")

figure, axes = plt.subplots(3, 1, sharex=True, figsize=(9, 9))
vaft.omas.plot_rogowski_coil_time_current(ods, ax=axes[0], show=False)
vaft.omas.plot_plasma_current_time(ods, ax=axes[1], show=False)
vaft.omas.plot_diamagnetic_flux_time(ods, ax=axes[2], show=False)
for axis in axes:
    axis.axvline(t_breakdown, color="k", ls="--", lw=0.8)
figure.tight_layout()
plt.show()

Compare the top panel with the middle one. The plasma-current sensor reads with the opposite sign to
$I_p$ — the shot-era sign convention is applied later in the chain — and it moves *before* the
breakdown line, by some ten kiloamperes: that is the current induced in the tungsten limiter while the
coils swing, which the sensor's contour links and the processed $I_p$ no longer contains, because the
FL10 compensation has taken it out. The diamagnetic sensor's stored current is an integrated,
calibrated signal of about thirteen amperes that changes by roughly half an ampere during the
discharge — it looks flat on a kiloampere axis — and the diamagnetic flux in the bottom panel is that
small change, left over once a reference waveform has been subtracted.

### Step 8 — startup proxies against time

`plot_startup_proxies_time` evaluates the three proxies of step 6 — $B_z$, $V_{\mathrm{loop}}$ and the
decay index — in the vacuum field at the startup reference point for every PF sample. Its default
window runs from the ohmic-coil onset to the plasma-current peak, and it marks the breakdown onset
itself.

In [ ]:
vaft.omas.plot_startup_proxies_time(ods)
plt.show()

The loop voltage climbs from its negative swing through zero, and the breakdown happens on its way
up; $B_z$ at the reference point comes back to within a few gauss of zero at breakdown — the vertical
field is weakest just when a null is wanted — and then grows to over a hundred gauss as the
vertical-field coils take over; the decay index sits low inside its band except early on, where $B_z$
changes sign and the ratio blows up.

How large should $B_z$ be once a current ring exists? The Shafranov estimate of step 6 answers that
for the peak current, with $l_i$ and $\beta_p$ assumed rather than reconstructed.

In [ ]:
from vaft.omas.process_wrapper import compute_startup_proxies_ods

ip_time = np.asarray(ods["magnetics.ip.0.time"], dtype=float)
ip_data = np.asarray(ods["magnetics.ip.0.data"], dtype=float)
t_ip_peak = float(ip_time[np.nanargmax(np.abs(ip_data))])
ip_peak = float(np.nanmax(np.abs(ip_data)))

assumed_li, assumed_beta_p = 1.0, 0.1  # assumptions for a young ohmic plasma, not reconstructed
b_v_needed = vaft.formula.startup.vertical_field_from_I_p_R0_a_beta_p_li(
    ip_peak, r_ref, minor_radius, assumed_beta_p, assumed_li)

proxies = compute_startup_proxies_ods(copy.deepcopy(ods), rz=(r_ref, 0.0))
b_z_vacuum = float(proxies["b_z"][int(np.argmin(np.abs(proxies["time"] - t_ip_peak)))])

print(f"I_p peak {ip_peak / 1e3:.1f} kA at {t_ip_peak * 1e3:.2f} ms ({(t_ip_peak - t_breakdown) * 1e3:+.1f} ms after breakdown)")
print(f"Shafranov estimate, R0 = {r_ref:.2f} m, a = {minor_radius:.2f} m: |B_v| = {abs(b_v_needed) * 1e4:.0f} G")
print(f"vacuum B_z at the reference point then: {b_z_vacuum * 1e4:+.0f} G")

Treat the comparison as an order-of-magnitude statement. The estimate assumes a thin circular ring
at large aspect ratio; VEST's aspect ratio is close to one. The vacuum $B_z$ is the coils' and the
vessel's field *without* the plasma — after breakdown the real vessel currents also respond to the
plasma — and $l_i$, $\beta_p$ are guesses. Agreement to within a factor of two is what this proxy
can promise.

#### The drive

The loop voltage is the time derivative of poloidal flux at a point — literally what the transformer
is doing to the plasma's future location. The field it corresponds to is

$$E_\varphi = -\frac{1}{2\pi R}\frac{\partial\psi}{\partial t},
  \qquad V_{\mathrm{loop}} = \oint \mathbf{E}\cdot\mathrm{d}\boldsymbol{\ell}
  = 2\pi R\,E_\varphi = -\frac{\partial\psi}{\partial t},$$

with $\psi$ in weber. Whether that $2\pi$ survives depends on whether the flux is stored per weber or
per radian, and VAFT's two kernels disagree on it *and* on the sign: `toroidal_electric_field` is the
expression above, while `loop_voltage_from_total_flux` is $+2\pi\,\mathrm{d}\psi_b/\mathrm{d}t$ on a
per-radian flux. That is [#354](https://github.com/VEST-Tokamak/vaft/issues/354), still open. The cell
below takes the first.

In [ ]:
time, v_loop = vaft.omas.compute_startup_loop_voltage_ods(ods, rz=(r_ref, 0.0))

figure, axes = plt.subplots()
axes.plot(time, v_loop)
axes.axvline(t_breakdown, color="k", ls="--", lw=0.8, label="breakdown onset")
axes.set_xlabel("Time [s]")
axes.set_ylabel(r"$V_{\mathrm{loop}}$ [V]")
axes.legend()
axes.grid(alpha=0.5)
plt.show()

v_loop_breakdown = float(np.interp(t_breakdown, time, v_loop))
print(f"peak |V_loop| = {np.nanmax(np.abs(v_loop)):.2f} V "
      f"at t = {time[np.nanargmax(np.abs(v_loop))] * 1e3:.2f} ms")
print(f"V_loop at breakdown = {v_loop_breakdown:.2f} V at R = {r_ref:.2f} m")

#### The vertical field and its decay index

A plasma ring wants to expand. The vertical field pushes back, and whether that restoring force is
*stable* depends on how fast the field falls off with radius. Both quantities come out of the same
flux map:

$$B_Z = -\frac{k}{R}\frac{\partial\psi}{\partial R},
  \qquad n = -\frac{R}{B_Z}\frac{\partial B_Z}{\partial R},
  \qquad k = \frac{\sigma_{R\varphi Z}\,\sigma_{B_p}}{(2\pi)^{e_{B_p}}},$$

where $k$ carries the COCOS orientation and the $2\pi$ together. So $n$ is a second derivative of
$\psi$ wearing a disguise — which is why it is the noisiest quantity in this session, and why it has
no value at all on the surface where $B_Z$ changes sign.

The window $0 < n < 1.5$ is the rigid-ring result, and each edge guards a different direction. Below
zero the field lines curve the wrong way and a ring nudged up or down keeps going: **vertically
unstable**. Above $1.5$ the vertical field falls off so fast that a ring pushed outward finds too little
field to push it back: **radially unstable**. It assumes a thin
current ring and no conducting wall. VEST has a very conducting wall — the same one solved for above
— so the true window is wider than the band drawn below. The cut is taken at two instants the data
named: the breakdown onset and the plasma-current peak.

In [ ]:
figure, axes = plt.subplots()
axes.axhspan(0.0, 1.5, alpha=0.15, label="rigid-ring band 0 < n < 1.5")
for name, instant in (("breakdown onset", t_breakdown), ("I_p peak", t_ip_peak)):
    radius, decay_index = vaft.omas.compute_decay_index_ods(ods, time=instant)
    finite = np.isfinite(decay_index)
    axes.plot(radius, decay_index, label=f"{name}, {instant * 1e3:.1f} ms")
    print(f"{name:15s} ({instant * 1e3:.2f} ms): n spans {decay_index[finite].min():.2f} "
          f"to {decay_index[finite].max():.2f}; entirely inside 0 < n < 1.5: "
          f"{bool(np.all((decay_index[finite] > 0.0) & (decay_index[finite] < 1.5)))}")
axes.set_xlabel("R [m]")
axes.set_ylabel("decay index $n$")
axes.legend()
axes.grid(alpha=0.5)
plt.show()

The two instants tell different stories, and both are right. At the breakdown onset the cut leaves
the band at large radius: the vertical field there is only a few gauss and falls off steeply, so $n$
climbs to several. But at breakdown there is no current ring to hold yet, so the question the decay
index answers has not started. By the plasma-current peak the vertical-field coils have taken over and
the index is inside the band across the whole cut, topping out near one at the outer end — a ring
there would be held rather than flung out.
Note also that $n$ is undefined where $B_z$ crosses zero; the calculation returns `nan` there rather
than a large meaningless number.

### Step 9 — the midplane, radially

The proxies above live at one point. `plot_vacuum_field_midplane` reads the same vacuum field along
the whole $Z = 0$ row inside the limiter: the loop voltage, $B_z$, the breakdown figure of merit
$E_\varphi B_\varphi / B_p$ with the empirical **1000 V/m Ohmic** and **100 V/m ECH-assisted**
thresholds drawn in, and the Lloyd margin (drive over the Townsend threshold, from a traced connection
length). Every panel carries the 2.45 GHz resonance radius $R_{\mathrm{ECR}}$ as a dash-dotted line.

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(12, 8))
for axis, field in zip(axes.ravel(), ("v_loop", "b_z", "breakdown", "lloyd_margin")):
    vaft.omas.plot_vacuum_field_midplane(ods, field=field, time=t_breakdown, ax=axis, show=False)
    axis.title.set_fontsize("small")
figure.tight_layout()
plt.show()

In [ ]:
# interaction_backend="auto" gives live time and field selectors in Jupyter. This cell drives the
# same controls from code: the four fields at breakdown, then the breakdown figure of merit a
# millisecond either side of it.
result = vaft.omas.plot_vacuum_field_midplane(
    ods, field="v_loop", interactive=True, interaction_backend="none",
)
controls = {control.name: control for control in result.controls}
print(f"{controls['time_index'].label}: {controls['field'].options}")

pf_time = np.asarray(ods["pf_active.time"], dtype=float)
around = int(np.argmin(np.abs(pf_time - t_breakdown)))
step = max(1, int(round(1e-3 / float(np.median(np.diff(pf_time))))))

result.state.set("time_index", around)
for field in ("v_loop", "b_z", "breakdown", "lloyd_margin"):
    result.state.set("field", field)
    curve = result.axes.get_lines()[0].get_ydata()
    print(f"  {result.axes.get_title()}: max {np.nanmax(curve):.3g}")
result.state.set("field", "breakdown")
for index in (around - step, around, around + step):
    result.state.set("time_index", index)
    curve = result.axes.get_lines()[0].get_ydata()
    print(f"  {result.axes.get_title()}: max {np.nanmax(curve):.3g}")
plt.close(result.figure)

Read what moves. The loop voltage rises gently towards the outboard side. $B_z$ is only a few gauss
across most of the row and strongest against the centre stack. The figure of merit
$E_\varphi B_\varphi / B_p$ is highest where $B_p$ is weakest: on this row it clears the 100 V/m
ECH-assisted threshold everywhere, and peaks just short of the 1000 V/m Ohmic one — this was an
EC-assisted startup, and it looks like one. A millisecond on either side of the onset the peak figure of
merit halves: the good-breakdown condition is a moment, not a state. The Lloyd margin is above one over
most of the row. The resonance line sits far outboard of the reference point, near the outboard limiter
— where the EC power seeds electrons is not where the null is best.

### Step 10 — the vacuum field in two dimensions

#### The field null

Breakdown needs somewhere for electrons to accelerate without promptly hitting a wall — a region of
weak poloidal field with long connection lengths. In the vacuum flux map it shows up as the null: the
other field component comes from the same map with the other derivative, and the null is where the
pair of them vanishes together,

$$B_R = \frac{k}{R}\frac{\partial\psi}{\partial Z},
  \qquad |B_p| = \sqrt{B_R^2 + B_Z^2} \longrightarrow 0 .$$

It is not a point of zero *total* field. $B_\varphi$ is untouched by everything above and is two
orders of magnitude larger there. A null is where the field becomes almost purely toroidal, so a line
wraps the torus many times before walking out to a wall — which is what buys the connection length
breakdown turns out to depend on. Read the contours for the shape of the startup field: a field null,
or a trapped-particle configuration (TPC), whose mirror-like curvature holds electrons in the
weak-field region instead of relying on a near-zero $B_p$ alone.

In [ ]:
vaft.omas.plot_equilibrium_field_psi_vacuum(ods, time=t_breakdown)
plt.show()

A single frame hides the most important fact about the null: it does not last. The canonical map,
`plot_vacuum_field`, draws six quantities of the coils' and the vessel's field from one evaluation —
the flux, $|B_p|$, the decay index, $|E_\varphi|$, a breakdown figure of merit and the Lloyd margin —
and caches the grid's response to the coils, so moving in time costs a contraction rather than a
solve. That is what makes it cheap enough to sweep.

Start with the single number that says how good a null is: how much of the vessel has a poloidal field
weaker than five gauss, over a few milliseconds either side of breakdown.

In [ ]:
from matplotlib.path import Path as MplPath

limiter = MplPath(np.column_stack([limiter_r, limiter_z]))
sweep_index = np.flatnonzero(np.abs(pf_time - t_breakdown) <= 2.5e-3)[::3]

weak_fraction = []
for index in sweep_index:
    field = vaft.omas.compute_vacuum_field_map(ods, time=float(pf_time[index]), resolution=33)
    mesh_r, mesh_z = np.meshgrid(field["r"], field["z"], indexing="ij")
    inside = limiter.contains_points(
        np.column_stack([mesh_r.ravel(), mesh_z.ravel()])).reshape(mesh_r.shape)
    b_p = vaft.formula.poloidal_field_magnitude(field["b_r"], field["b_z"])
    weak_fraction.append(np.mean(b_p[inside] < 5e-4))
weak_fraction = np.asarray(weak_fraction)
sweep_time = pf_time[sweep_index]
t_null = float(sweep_time[np.argmax(weak_fraction)])

figure, axes = plt.subplots()
axes.plot(sweep_time * 1e3, weak_fraction * 100)
axes.axvline(t_breakdown * 1e3, color="k", ls="--", lw=0.8, label="onset, from the light")
axes.set_xlabel("Time [ms]")
axes.set_ylabel(r"vessel with $|B_p| < 5$ G [%]")
axes.legend()
axes.grid(alpha=0.5)
plt.show()

print(f"widest null   : {t_null * 1e3:.2f} ms, {weak_fraction.max():.0%} of the vessel below 5 G")
print(f"light appears : {(t_breakdown - t_null) * 1e3:+.2f} ms after it")

The null is not an object the field has; it is a moment the field passes through. For about a
millisecond a large part of the vessel has almost no poloidal field, and then it closes again. The
widest point comes a fraction of a millisecond *before* the light — which is the right order. The field
opens, the gas inside the opening avalanches, and the H-alpha detector sees the result. Nothing forced
the two curves to line up: the onset was timed from the light alone, and this curve comes from the
coils alone.

#### Where the 2.45 GHz source resonates

An electron gyrates at $f_{ce} = eB/(2\pi m_e)$, so the 2.45 GHz magnetron is absorbed where
$|B| = 2\pi m_e f / e = 0.0875$ T. Before a plasma exists $|B|$ is essentially the toroidal field, which
falls as $1/R$, so the fundamental resonance is a vertical line at
$R_{\mathrm{ECR}} = |B_T R| / B_{\mathrm{ECR}}$. Every map below draws it when given
`ec_frequency_Hz`.

In [ ]:
EC_FREQUENCY_HZ = 2.45e9
B_ECR = vaft.formula.startup.electron_cyclotron_resonance_field(EC_FREQUENCY_HZ)
R_ECR = float(vaft.formula.startup.electron_cyclotron_resonance_radius(b_t_r, EC_FREQUENCY_HZ))
print(f"B_ECR = {B_ECR * 1e3:.1f} mT; with B_T R = {abs(b_t_r):.4f} T m at breakdown, "
      f"R_ECR = {R_ECR:.3f} m")
print(f"limiter spans R = {limiter_r.min():.3f} - {limiter_r.max():.3f} m: "
      f"the resonance is {'inside' if limiter_r.min() < R_ECR < limiter_r.max() else 'outside'} the vessel")

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(10, 7))
for axis, quantity in zip(axes, ("psi", "b_poloidal")):
    vaft.omas.plot_vacuum_field(ods, field=quantity, time=t_null, resolution=65,
                                ec_frequency_Hz=EC_FREQUENCY_HZ, ax=axis, show=False)
figure.suptitle(f"The vacuum field at the widest null, t = {t_null * 1e3:.1f} ms")
plt.show()

Now the quantities that decide whether that null is *used*, at the breakdown onset: the drive
$|E_\varphi|$, the decay index, the figure of merit $E_\varphi B_\varphi / B_p$ and the Lloyd margin.

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(11, 14))
for axis, quantity in zip(axes.ravel(), ("e_toroidal", "decay_index", "breakdown", "lloyd_margin")):
    vaft.omas.plot_vacuum_field(ods, field=quantity, time=t_breakdown, resolution=33,
                                ec_frequency_Hz=EC_FREQUENCY_HZ, ax=axis, show=False)
    axis.title.set_fontsize("small")
figure.suptitle(f"Breakdown quantities at the onset, t = {t_breakdown * 1e3:.1f} ms")
plt.show()

In [ ]:
# interaction_backend="auto" gives a live time slider in Jupyter. This cell drives the
# same control from code, so it also runs headless and prints what the slider shows.
result = vaft.omas.plot_vacuum_field(
    ods, field="b_poloidal", resolution=33,
    interactive=True, interaction_backend="none",
)
slider = next(control for control in result.controls if control.name == "time_index")
print(f"{slider.label}: {slider.options[1] + 1} positions")
for index in (around - 30, around, around + 30):
    result.state.set("time_index", int(index))
    print(f"  {result.axes.get_title()}")
plt.close(result.figure)

#### Was that drive enough?

"Weak poloidal field and long connection lengths" is a description, not a test. Whether an electron
avalanches before it reaches a wall turns on three numbers: how hard it is pushed ($E_\parallel$), how
much gas there is to ionise ($p$), and how far it travels before hitting something ($L$). The pressure
was plotted in step 4. The prefill is the median of the gauge over the 20 ms before the onset
(`compute_prefill_pressure_ods`) — the window matters, because over everything before breakdown the
base vacuum ahead of the gas puff would win.

In [ ]:
if "barometry" in ods:
    pressure = np.asarray(ods["barometry.gauge.0.pressure.data"], dtype=float)
    pressure_time = np.asarray(ods["barometry.gauge.0.pressure.time"], dtype=float)
    prefill = vaft.omas.compute_prefill_pressure_ods(ods, before=t_breakdown)
    near_onset = np.abs(pressure_time - t_breakdown) <= 5e-3
    at_onset = float(np.nanmedian(pressure[near_onset])) if near_onset.any() else float("nan")
    molecules = vaft.formula.neutral_density_from_pressure(prefill)

    figure, axes = plt.subplots()
    axes.plot(pressure_time * 1e3, pressure * 1e3)
    axes.axvspan((t_breakdown - 2e-2) * 1e3, t_breakdown * 1e3, alpha=0.15, label="prefill window")
    axes.axvline(t_breakdown * 1e3, color="k", ls="--", lw=0.8, label="breakdown onset")
    axes.set_xlim((t_breakdown - 1e-1) * 1e3, (t_breakdown + 1e-1) * 1e3)
    axes.set_xlabel("Time [ms]")
    axes.set_ylabel("Pressure [mPa]")
    axes.legend()
    axes.grid(alpha=0.5)
    plt.show()

    print(f"prefill (median of the 20 ms before breakdown): {prefill:.3e} Pa "
          f"= {prefill / vaft.formula.PA_PER_TORR:.2e} Torr")
    print(f"H2 molecules: {molecules:.2e} m^-3, "
          f"hydrogen atoms: {vaft.formula.atomic_inventory_from_molecular_gas(molecules):.2e} m^-3")
    print(f"the same gauge within 5 ms of the breakdown instant: {at_onset:.3e} Pa "
          f"({at_onset / prefill:.2f}x the prefill)")
else:
    prefill = float("nan")
    print(f"shot {SHOT} carries no barometry: the Lloyd cells below have no pressure to work with")

The gauge never settles. It rises through the gas puff and peaks *after* breakdown, as the discharge
drives gas off the wall faster than a Penning gauge can follow, so "the prefill" is a choice of window
rather than a reading — worth remembering when the numbers below start looking precise.

The third quantity is not in the file. Before tracing it, look at the estimate everyone reaches for
first, because it is worth knowing how far it can be trusted. A line at pitch $B_\perp / B_T$ needs
$B_T / B_\perp$ turns to drift across a region of size $a$, which gives

$$L_{\mathrm{open}} \sim a\,\frac{B_T}{B_\perp}.$$

No wall in it, no null geometry, no constant of order one. It is worth plotting for its *slope* — what
a better null buys — and the trace below says how far to trust it as a number.

In [ ]:
b_toroidal = abs(b_t_r) / r_ref  # B_T at the startup reference point, at breakdown
b_perp = np.geomspace(1e-4, 5e-3, 200)  # 1 G to 50 G at the null
length_open = minor_radius * b_toroidal / b_perp
e_threshold = vaft.formula.lloyd_breakdown_field(prefill, length_open)
e_measured = abs(v_loop_breakdown) / (2 * np.pi * r_ref)

figure, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].loglog(b_perp * 1e4, length_open)
axes[0].set_ylabel(r"$L_{\mathrm{open}}$ [m]")
axes[1].loglog(b_perp * 1e4, e_threshold)
axes[1].axhline(e_measured, color="k", ls="--", lw=0.8)
axes[1].set_ylabel(r"$E_{\mathrm{BD}}$ [V/m]")
axes[1].set_xlim(axes[0].get_xlim())  # so the right curve visibly stops early
for axis in axes:
    axis.set_xlabel(r"$B_\perp$ at the null [G]")
    axis.grid(alpha=0.5, which="both")
plt.show()

print(f"B_T at R = {r_ref:.2f} m: {b_toroidal:.4f} T")
print(f"drive at breakdown at the same radius: E = {e_measured:.3f} V/m")

It no longer has to be assumed. `trace_field_line` follows a line through a field and stops it at the
wall; what it could not do before breakdown was get a field to follow, because the only field it knew
how to build came from an equilibrium, and a discharge that has not broken down has none.
`compute_connection_length_map_ods` builds the vacuum field instead and traces a line from every point
of a grid, in both directions, to the limiter. It takes a few seconds, which is why it is a one-off
here and not a slider frame.

In [ ]:
traced = vaft.omas.compute_connection_length_map_ods(ods, time=t_breakdown, resolution=33)
interior = ~traced["outside"]
open_lines = interior & ~traced["saturated"]

field = vaft.omas.compute_vacuum_field_map(ods, time=t_breakdown, resolution=33)
b_p = vaft.formula.poloidal_field_magnitude(field["b_r"], field["b_z"])
scaling = minor_radius * (abs(b_t_r) / field["r"][:, None]) / b_p
ratio = scaling[open_lines] / traced["length_m"][open_lines]

figure, axes = plt.subplots()
axes.loglog(traced["length_m"][open_lines], scaling[open_lines], ".", ms=3, alpha=0.5)
axes.plot([1.0, 300.0], [1.0, 300.0], "k--", lw=0.8, label="scaling = trace")
axes.set_xlabel("traced connection length [m]")
axes.set_ylabel(r"$a B_T / B_\perp$ scaling [m]")
axes.legend()
axes.grid(alpha=0.5, which="both")
plt.show()

print(f"traced from {interior.sum()} points; {traced['saturated'].sum()} never met the wall")
print(f"scaling / trace: median {np.median(ratio):.2f}, "
      f"half of them between {np.percentile(ratio, 25):.2f} and {np.percentile(ratio, 75):.2f}")
print(f"log-log correlation: {np.corrcoef(np.log(scaling[open_lines]), np.log(traced['length_m'][open_lines]))[0, 1]:.2f}")

Both curves of the scaling figure stop, and that is not a plotting failure. Below $A p L = 1$ the
avalanche cannot close *at any field*, so `lloyd_breakdown_field` warns and returns `nan` instead of a
large number — the model has a domain, and its edge is a physical statement. At this prefill, a
connection length shorter than about a hundred metres means no breakdown however hard you push.

Turn the picture around and hold $L$ fixed instead. The threshold is then a curve in pressure with a
minimum: too little gas and there is nothing to ionise, too much and the electrons collide before they
have gained the ionisation energy. It is a Paschen curve with the electrode gap replaced by the
connection length.

In [ ]:
p_scan = np.geomspace(1e-3, 3e-2, 300)

figure, axes = plt.subplots()
for length in (100.0, 200.0, 400.0, 1000.0):
    axes.loglog(p_scan * 1e3, vaft.formula.lloyd_breakdown_field(p_scan, length),
                label=f"L = {length:.0f} m")
axes.axhline(e_measured, color="k", ls="--", lw=0.8)
axes.axvline(prefill * 1e3, color="k", ls=":", lw=0.8)
axes.set_xlabel("Prefill pressure [mPa]")
axes.set_ylabel(r"$E_{\mathrm{BD}}$ [V/m]")
axes.legend()
axes.grid(alpha=0.5, which="both")
plt.show()

length_scan = np.linspace(80.0, 1000.0, 46001)
threshold = vaft.formula.lloyd_breakdown_field(prefill, length_scan)
margin = vaft.formula.breakdown_margin(e_measured, threshold)
enough = length_scan[margin >= 1.0]
doubled = length_scan[vaft.formula.breakdown_margin(2 * e_measured, threshold) >= 1.0]

if np.isfinite(threshold).any():
    print(f"no threshold exists at all below L = {length_scan[np.isfinite(threshold)].min():.2f} m")
if enough.size:
    print(f"the drive at breakdown is enough from L = {enough.min():.2f} m, "
          f"i.e. a null better than {minor_radius * b_toroidal / enough.min() * 1e4:.2f} G")
else:
    print(f"the drive at breakdown clears no connection length up to {length_scan.max():.0f} m")
print(f"margin at L = 100 m: {np.interp(100.0, length_scan, margin):.2f}, "
      f"at L = 150 m: {np.interp(150.0, length_scan, margin):.2f}")
if enough.size and doubled.size:
    print(f"doubling the drive would only move the requirement to {doubled.min():.2f} m")
print(f"the fit is being read at E/p = "
      f"{e_measured / (prefill / vaft.formula.PA_PER_TORR) / 100:.0f} V/cm/Torr")

In [ ]:
e_local = np.abs(vaft.formula.toroidal_electric_field(field["r"][:, None], field["dpsi_dt"]))
threshold_map = vaft.formula.lloyd_breakdown_field(prefill, traced["length_m"])
margin_map = vaft.formula.breakdown_margin(e_local, threshold_map)
has_threshold = interior & np.isfinite(margin_map)
print(f"a threshold exists over {has_threshold.sum() / interior.sum():.0%} of the vessel")
if has_threshold.any():
    print(f"where one exists, the drive clears it at {np.mean(margin_map[has_threshold] >= 1):.0%} of the points")

# The same numbers, and the rest of this step, in one call: what Part II tabulates per shot.
summary = vaft.omas.startup_summary(ods)
print(f"startup_summary: threshold fraction {summary['lloyd_threshold_fraction']:.0%}, "
      f"clears it at {summary['lloyd_margin_fraction']:.0%}, null area {summary['null_area_fraction']:.0%}, "
      f"R_ECR {summary['r_ecr_m']:.3f} m")

Read those numbers together and the connection length turns out to be the whole answer. At this prefill
and the drive at breakdown, the avalanche becomes possible at a connection length a little over a hundred
metres — on 39915 about 111 m — which the scaling above translates into a null better than about four and
a half gauss. Every one of those figures is soft: the gauge keeps rising through the breakdown instant, the
$E/p$ being read is extreme, and the traced lengths scatter around the scaling by tens of percent.

One caveat is not about the data at all. Lloyd's constants are a two-parameter fit to hydrogen over a
band of $E/p$, and the printed $E/p$ above sits at the top of where such fits are usually quoted — so the
threshold here is an extrapolation, and it gets worse, not better, as the curve approaches its domain
edge. That is a reason to trust the *ordering* of operating points and not the third significant figure.

What does not move is *which* quantity decides, and the Lloyd-margin map above shows it directly. Where a
threshold exists at all the drive clears it almost everywhere; what decides breakdown is whether a line
runs far enough for a threshold to exist, and that is the blank part of the map — part of the answer, not
missing data. Doubling the loop voltage buys only a dozen metres, because the drive only enters through a
logarithm; halving $B_\perp$ doubles the connection length outright.

### Step 11 — the camera and the vacuum field lines at breakdown

Back to the camera, now with the field it saw. The lines drawn over the frame below are **vacuum field
lines** — traced through the coils' and the vessel's field at the time of the camera frame nearest the breakdown
onset, never an EFIT reconstruction, which does not exist yet. They are the paths an electron born at
the startup reference point, or on the 2.45 GHz resonance layer, would follow along $\mathbf{B}$ for
up to one toroidal turn each way from its seed,
projected through the camera's calibrated pose. The cell reloads the packaged frames nearest breakdown
and nearest the plasma-current peak.

In [ ]:
if camera_frames:
    all_times = np.asarray([time_s for time_s, _ in camera_frames])
    chosen = sorted({int(np.argmin(np.abs(all_times - instant))) for instant in (t_breakdown, t_ip_peak)})
    images = [cv2.imread(str(camera_frames[i][1]), cv2.IMREAD_GRAYSCALE) for i in chosen]
    del ods["camera_visible"]
    vfit_camera_visible_static(
        ods, lines_n=images[0].shape[0], columns_n=images[0].shape[1], channel_name="Fast Camera",
    )
    vfit_camera_visible_dynamic(ods, images=images, times_s=all_times[chosen].tolist())

    vaft.omas.plot_camera_visible_image_vacuum_field_line(
        ods, shot=SHOT, time=t_breakdown, seeds=[(r_ref, 0.0), (R_ECR, 0.0)], max_turns=1,
    )
    plt.show()
    print(f"frame nearest breakdown: {all_times[chosen[0]] * 1e3:.1f} ms; "
          f"seeds at R = {r_ref:.3f} m (reference point) and R = {R_ECR:.3f} m (2.45 GHz ECR)")
else:
    print(f"no packaged camera frames for shot {SHOT}: no field-line overlay")

For contrast, and clearly **after** the startup: once a plasma equilibrium has been reconstructed
(session 03), the same camera frame can carry the EFIT last closed flux surface and magnetic axis
instead. The packaged equilibrium only starts some ten milliseconds into the discharge, so this overlay
is drawn at the plasma-current peak — nothing here reconstructs the breakdown itself.

In [ ]:
if camera_frames and "equilibrium" in ods:
    frame_times = np.asarray(ods["camera_visible.time"], dtype=float)
    overlay_index = int(np.argmin(np.abs(frame_times - t_ip_peak)))
    vaft.omas.plot_camera_visible_image_efit_overlay(ods, shot=SHOT, frame_index=overlay_index)
    plt.show()
    print(f"frame at {frame_times[overlay_index] * 1e3:.1f} ms, with the wall, the last closed flux "
          f"surface and the magnetic axis projected through the calibrated pose")
else:
    print(f"shot {SHOT}: no camera frames or no equilibrium, so no EFIT overlay")

The two overlays answer different questions. The vacuum lines say where the *field* would carry seed
electrons at the moment the gas broke down — from the reference point and from the resonance layer, a
turn around the torus. The EFIT contour says where the *plasma* sat once it existed. Matching the light
in the first frame to the vacuum lines, not to the equilibrium, is the startup-physics reading of the
camera.

## Interpretation Checkpoints

Each of these has a definite answer in what you have already printed or plotted.

1. **`decided by` and `light and current`.** Which diagnostic set `t_breakdown`, and how far apart did
   light and current start? Would you trust the plasma current alone to define "the plasma started"?
2. **The widest null and the light.** The sweep printed the widest-null time and how long after it the
   light appeared. Is that the order causality demands? What would it mean if the sign were reversed?
3. **The vessel current opposes the coil current.** In the current overview, when is the vessel
   contribution largest, and what is the transformer doing then?
4. **The two decay-index cuts.** One printed `entirely inside 0 < n < 1.5: False`, the other `True`.
   Why does only one of them bear on stability?
5. **The Lloyd numbers.** A threshold exists over only part of the vessel, yet where it exists the drive
   clears it almost everywhere. Which knob — loop voltage, prefill, or null quality — would you turn to
   make a marginal breakdown robust, and why?
6. **`compute_eddy_currents` unlocked plots.** Why should a *modelling* step change what you are allowed
   to plot?

## Integrated Analysis

### Part II — the same analysis on several discharges

A single shot tells a story; a set of shots tells you which parts of it are the operating point and
which are luck. Everything in Part I was written without a shot-specific number, so it runs unchanged
on a list. Three discharges ship with VAFT: 39915, and two later shots, 41524 and 41672, with the EC
source off. The commented line is the lab-mode path for any list of shots.

In [ ]:
shots = [39915, 41524, 41672]
ods_list = [vaft.omas.sample_ods(shot) for shot in shots]
# ods_list = vaft.database.load(shots)  # lab mode: any list of shots, needs HSDS credentials
for case in ods_list:
    vest_wall(case)
print(f"loaded {len(ods_list)} discharges: {shots}")

Start with the table. `startup_summary` gathers what Part I read off one shot — the onset and what
decided it, the current peak, the prefill, the proxies at the reference point, the null area, the Lloyd
fractions, the resonance radius and the EC power at breakdown — into one flat `dict`, on a private copy,
with `None` and a reason wherever a shot lacks an input. A coarse `resolution` keeps three shots quick.

In [ ]:
rows = []
for shot, case in zip(shots, ods_list):
    row = vaft.omas.startup_summary(case, resolution=17)
    rows.append(row)
    for key, reason in row["unavailable"].items():
        print(f"  {shot}: {key} unavailable ({reason})")

columns = [
    ("shot", "shot", 1.0, "{:.0f}"),
    ("t_bd [ms]", "t_breakdown", 1e3, "{:.2f}"),
    ("source", "onset_source", None, "{}"),
    ("Ip peak [kA]", "ip_peak_A", 1e-3, "{:.0f}"),
    ("prefill [mPa]", "pressure_Pa", 1e3, "{:.2f}"),
    ("V_loop [V]", "v_loop_V", 1.0, "{:.2f}"),
    ("B_z [G]", "b_z_T", 1e4, "{:.1f}"),
    ("n", "decay_index", 1.0, "{:.2f}"),
    ("null <5G", "null_area_fraction", 1e2, "{:.0f}%"),
    ("Lloyd exists", "lloyd_threshold_fraction", 1e2, "{:.0f}%"),
    ("R_ECR [m]", "r_ecr_m", 1.0, "{:.3f}"),
    ("EC [kW]", "ec_power_W", 1e-3, "{:.2f}"),
]
table = [
    [
        "-" if row[key] is None else (fmt.format(row[key]) if scale is None else fmt.format(row[key] * scale))
        for _, key, scale, fmt in columns
    ]
    for row in rows
]
try:
    import pandas as pd
    print(pd.DataFrame(table, columns=[name for name, *_ in columns]).to_string(index=False))
except ImportError:
    widths = [max(len(name), *(len(line[i]) for line in table)) for i, (name, *_) in enumerate(columns)]
    print("  ".join(name.rjust(width) for (name, *_), width in zip(columns, widths)))
    for line in table:
        print("  ".join(cell.rjust(width) for cell, width in zip(line, widths)))

onsets = [row["t_breakdown"] for row in rows if row["t_breakdown"] is not None]
peaks = [row["t_ip_peak"] for row in rows if row["t_ip_peak"] is not None]

Every line plot of Part I accepts a list and overlays it, one colour per shot. The response first, on
the absolute DAQ clock, over a window the table's onsets and peaks define.

In [ ]:
figure, axes = plt.subplots(3, 1, sharex=True, figsize=(9, 10))
vaft.omas.plot_plasma_current_time(ods_list, ax=axes[0], show=False)
vaft.omas.plot_spectrometer_uv_time_intensity(ods_list, emission="H_alpha", ax=axes[1], show=False)
vaft.omas.plot_flux_loop_time_voltage(ods_list, selection="inboard_mid", ax=axes[2], show=False)
if onsets and peaks:
    axes[-1].set_xlim(min(onsets) - 2e-2, max(peaks) + 2e-2)
figure.tight_layout()
plt.show()

In [ ]:
with_ec = [case for case in ods_list if "ec_launchers" in case]
with_gauge = [case for case in ods_list if "barometry" in case]
figure, axes = plt.subplots(3, 1, sharex=True, figsize=(9, 10))
vaft.omas.plot_tf_coil_time_current(ods_list, ax=axes[0], show=False)
if with_ec:
    vaft.omas.plot_ec_launchers_time_power(with_ec, smooth=1e-3, ax=axes[1], show=False)
else:
    print("no shot carries ec_launchers")
if with_gauge:
    vaft.omas.plot_barometry_time_pressure(with_gauge, ax=axes[2], show=False)
else:
    print("no shot carries barometry")
figure.tight_layout()
plt.show()

`plot_startup_proxies_time` takes the list too, marking each shot's own breakdown onset in its colour.
Its default window is the *first* shot's; a window wide enough for all of them comes from the table.

In [ ]:
if onsets and peaks:
    vaft.omas.plot_startup_proxies_time(ods_list, time_range=(min(onsets) - 1e-2, max(peaks)))
else:
    vaft.omas.plot_startup_proxies_time(ods_list)
plt.show()

The shots broke down at different DAQ times, which smears every overlay. `change_time_convention`
shifts every time-like leaf of an ODS so that a chosen event is zero — here the breakdown onset. It
works **in place** (on one ODS, or on an ODC), so align copies and keep the originals on the DAQ clock.

In [ ]:
aligned = [copy.deepcopy(case) for case in ods_list]
for case in aligned:
    vaft.omas.change_time_convention(case, "breakdown")

figure, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 7))
vaft.omas.plot_plasma_current_time(aligned, ax=axes[0], show=False)
vaft.omas.plot_spectrometer_uv_time_intensity(aligned, emission="H_alpha", ax=axes[1], show=False)
for axis in axes:
    axis.axvline(0.0, color="k", ls="--", lw=0.8)
axes[1].set_xlim(-2e-2, 4e-2)
axes[1].set_xlabel("Time since breakdown onset [s]")
figure.tight_layout()
plt.show()

Read the table as an operating-space comparison, and keep the caveats attached.

- **39915** broke down with about 3 kW of net EC power, a null covering roughly a third of the vessel,
  a Lloyd threshold over nearly half of it, and $B_z$ within a few gauss of zero at the reference point.
- **41524 and 41672** broke down with the EC source off, went on to much larger currents, and did so
  with a stronger vertical field at the reference point and almost no vessel below 5 G at their onsets.
- On both later shots the gauge still reads the **base vacuum** in the 20 ms before breakdown — their
  gas reaches the gauge only after the onset — so no Lloyd threshold exists anywhere. That does not mean
  breakdown was impossible; it means the prefill window missed the gas.
- 41672's vacuum loop voltage at the reference point is a fraction of a volt at its onset. Before reading
  that as physics, run step 7's model check on that shot: a proxy is only as good as the vacuum model
  under it.

That is the most useful thing a comparison table does — it shows which numbers are physics and which are
the definition of a window.

## Independent Exercise

### A one-page startup report from the VEST log

Use the real VEST experiment log and the VAFT database (lab mode).

1. **Browse** the experiment log for a date with a startup scan or clearly different operating
   conditions — a prefill scan, EC on/off, a PF5 null change, a TF change.
2. **Pick a small shot set** (two to five shots) that isolates one change, and load it with
   `vaft.database.load(shots)`.
3. **Run Part I and Part II** on it: breakdown timing, actuators, the vacuum-model check, proxies,
   the midplane cut, the null sweep, the Lloyd margin, `startup_summary` for the table.
4. **Write a one-page report** with these sections:
   - *Date and purpose* — what the session on that day was trying to do;
   - *Shots and why* — which shots, and what distinguishes them;
   - *Actuator conditions* — TF, PF waveforms, prefill, EC power;
   - *Breakdown timing evidence* — `t_breakdown`, its source, light/current agreement;
   - *Vacuum-field and null conditions* — model check quality, widest null, connection length, Lloyd
     margin;
   - *Proxy interpretation* — $V_{\mathrm{loop}}$, $B_z$, decay index, with their caveats;
   - *Cross-shot comparison* — one figure or table;
   - *Limitations and missing diagnostics* — unmapped signals (gas valve), gauge lag, the #354
     convention, the reduced-model assumptions.

### What #230 asked for, and where it stands

Issue [#230](https://github.com/VEST-Tokamak/vaft/issues/230) asked for two quantities, and this session
now computes both. **Connection length** is traced from every point of the vessel through the vacuum
field; what it does not do is tell a line that never meets the wall apart from one that is merely long —
both stop at the tracing limit and are flagged `saturated`. **The 2.45 GHz EC resonance layer** is placed
from $B_T R$, and the EC power that makes it matter is mapped (#165); the launch geometry is not yet.

In [ ]:
# Lab mode: needs HSDS credentials. Replace the shots with the set you chose from the log.
# shots = [<shot_a>, <shot_b>, <shot_c>]
# cases = vaft.database.load(shots)
#
# for case in cases:
#     row = vaft.omas.startup_summary(case)
#     print(row["shot"], row["t_breakdown"], row["pressure_Pa"], row["lloyd_threshold_fraction"])
#
# vaft.omas.plot_startup_proxies_time(cases)
# plt.show()

## Takeaways and Next Steps

- A discharge is a sequence — breakdown, burn-through, ramp-up — and each phase shows up in a different
  diagnostic. Find the breakdown time once, from the data, and read everything against it.
- VEST's PF waveforms are capacitor-bank discharges; PF1 supplies the inductive drive, PF5 shapes the
  null, PF6 and PF9/10 hold the ring radially. The EC power is now a signal, not an assumption.
- One Green function builds the mutual-inductance matrices, the vessel eddy currents, the synthetic
  magnetics and the vacuum-field maps. The vessel is part of the circuit, and every magnetic measurement
  sees its current.
- Check a model against what was measured before deriving anything from it.
- $V_{\mathrm{loop}}$, $B_z$ and the decay index at the startup reference point are reduced proxies with
  stated assumptions; the null, the connection length and the Lloyd margin decide breakdown, and the
  connection length decides it most.
- A comparison table is only as good as its definitions — a prefill window can decide a Lloyd verdict.

**Next**: Session 03 takes the discharge past startup and reconstructs the equilibrium it settled into —
the third use of the Green-function machinery, with a plasma current in the sum.